In [1]:
!pip install pandas numpy matplotlib scikit-learn lifelines openpyxl

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.0/350.0 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 9.2 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4030 sha256=7b875f696af0bc1a0bf10f3ee7ce2c24b7abfa6b99f1ab0a7de61e70087f54c2
  Stored in directory: /root/.cache/pip/wheels/50/37/21/0a719b9d89c635e89ff24bd93b862882ad675279552013b2fb
Successfully built autograd-gamma


In [2]:
# ==========================================
# CELL 1: Install & Import Libraries (Cox)
# ==========================================

# Uncomment if first time running
# !pip install pandas numpy matplotlib scikit-learn lifelines openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# sklearn utilities
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

# Cox model library
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

import warnings
warnings.filterwarnings("ignore")

# ==========================================
# Global configuration 
# ==========================================

N_REPEATS = 25
RANDOM_STATE_BASE = 42

print("Enhanced Cox Survival Pipeline Initialized")

Enhanced Cox Survival Pipeline Initialized


In [4]:
# ==========================================
# CELL 2: Load & Preprocess Data
# ==========================================

file_path = r"/kaggle/input/datasets/sweta2000/umhs-dataset/Training.xlsx"

data = pd.read_excel(file_path)

# Encode categorical variables
data["Race"] = np.where(data["Race"] == "Caucasian", 1, 0)
data["Smoking"] = np.where(data["Smoking"] == "Current", 1, 0)
data["Alcohol"] = np.where(data["Alcohol"] == "Yes", 1, 0)
data["Drug"] = np.where(data["Drug"] == "Yes", 1, 0)

# Convert numeric safely
data["HGB"] = pd.to_numeric(data["HGB"], errors="coerce")
data["Duration"] = pd.to_numeric(data["Duration"], errors="coerce")
data["EndEvent"] = pd.to_numeric(data["EndEvent"], errors="coerce")
data["MAGGIC"] = pd.to_numeric(data["MAGGIC"], errors="coerce")

# Create survival columns
data["time"] = data["Duration"]
data["event"] = data["EndEvent"]

data.drop(columns=["Duration", "EndEvent"], inplace=True)

data = data.loc[:, ~data.columns.duplicated()]

print("Data Loaded. Shape:", data.shape)


Data Loaded. Shape: (343, 157)


In [5]:
# ==========================================
# CELL 3: Create D, M, DM subsets (GBSA)
# ==========================================

# Get column list
cols = list(data.columns)

# Find index positions
idx_race = cols.index("Race")
idx_c_sa = cols.index("C_SA")
idx_rvpower_s = cols.index("RVpowerIndex_S")

# ------------------------------------------
# Digital features subset (D)
# ------------------------------------------
D = pd.concat([
    data.iloc[:, idx_race:idx_c_sa],
    data[["time", "event"]]
], axis=1)

# ------------------------------------------
# MRI features subset (M)
# ------------------------------------------
M = pd.concat([
    data.iloc[:, idx_c_sa:idx_rvpower_s + 1],
    data[["time", "event"]]
], axis=1)

# ------------------------------------------
# Combined features subset (DM)
# ------------------------------------------
DM = pd.concat([
    data.iloc[:, idx_race:idx_rvpower_s + 1],
    data[["time", "event"]]
], axis=1)

# ------------------------------------------
# Remove duplicate columns (important)
# ------------------------------------------
D = D.loc[:, ~D.columns.duplicated()]
M = M.loc[:, ~M.columns.duplicated()]
DM = DM.loc[:, ~DM.columns.duplicated()]

# ------------------------------------------
# Print shapes
# ------------------------------------------
print("D:", D.shape)
print("M:", M.shape)
print("DM:", DM.shape)

D: (343, 69)
M: (343, 82)
DM: (343, 149)


In [12]:
# ==========================================
# CELL 2: KNN Imputation for Cox datasets
# ==========================================

def impute_dataset(dataset, name):

    print(f"\nImputing {name}")

    imputer = KNNImputer(n_neighbors=5)

    feature_cols = dataset.columns.drop(["time", "event"])

    imputed_features = pd.DataFrame(
        imputer.fit_transform(dataset[feature_cols]),
        columns=feature_cols,
        index=dataset.index
    )

    dataset_imputed = pd.concat(
        [imputed_features, dataset[["time","event"]]],
        axis=1
    )

    missing = dataset_imputed.isna().sum().sum()

    print(f"{name} missing values after imputation: {missing}")

    return dataset_imputed


# Create imputed datasets
D_imputed  = impute_dataset(D,  "D")
M_imputed  = impute_dataset(M,  "M")
DM_imputed = impute_dataset(DM, "DM")


Imputing D
D missing values after imputation: 0

Imputing M
M missing values after imputation: 0

Imputing DM
DM missing values after imputation: 0


In [13]:
# ==========================================
# Updated Cox Variable Hunting Function
# ==========================================

from collections import defaultdict
from sklearn.preprocessing import StandardScaler
from lifelines import CoxPHFitter
import numpy as np

def cox_variable_hunting(dataset, model_name, n_repeats=25):

    print(f"\nStarting Variable Hunting for {model_name}")

    feature_names = list(dataset.columns)
    feature_names.remove("time")
    feature_names.remove("event")

    freq_counter = defaultdict(int)
    importance_counter = defaultdict(float)

    for rep in range(n_repeats):

        print(f"{model_name} Repeat {rep+1}/{n_repeats}")

        # ==========================================
        # Scale features (important for Cox stability)
        # ==========================================
        scaler = StandardScaler()

        X_scaled = pd.DataFrame(
            scaler.fit_transform(dataset[feature_names]),
            columns=feature_names,
            index=dataset.index
        )

        df_scaled = pd.concat(
            [X_scaled, dataset[["time","event"]]],
            axis=1
        )

        # ==========================================
        # Train Cox with regularization
        # ==========================================
        cph = CoxPHFitter(
            penalizer=0.1,
            l1_ratio=0.5
        )

        try:

            cph.fit(
                df_scaled,
                duration_col="time",
                event_col="event"
            )

        except:

            print("Skipping unstable iteration")
            continue

        # ==========================================
        # Use absolute coefficient as importance
        # ==========================================
        coef = cph.params_.abs()

        importance_dict = coef.to_dict()

        threshold = np.mean(list(importance_dict.values()))

        selected_features = [
            feature for feature, imp in importance_dict.items()
            if imp >= threshold
        ]

        for feature in selected_features:

            freq_counter[feature] += 1
            importance_counter[feature] += importance_dict[feature]

    # ==========================================
    # Average importance
    # ==========================================
    avg_importance = {
        feature: importance_counter[feature] / n_repeats
        for feature in importance_counter
    }

    sorted_features = sorted(
        avg_importance.items(),
        key=lambda x: x[1],
        reverse=True
    )

    sorted_features_dict = dict(sorted_features)

    print(f"\nCompleted Variable Hunting for {model_name}")

    return {
        "frequency": dict(freq_counter),
        "importance": sorted_features_dict
    }

In [14]:
cox_results_D  = cox_variable_hunting(D_imputed,  "D")
cox_results_M  = cox_variable_hunting(M_imputed,  "M")
cox_results_DM = cox_variable_hunting(DM_imputed, "DM")


Starting Variable Hunting for D
D Repeat 1/25
D Repeat 2/25
D Repeat 3/25
D Repeat 4/25
D Repeat 5/25
D Repeat 6/25
D Repeat 7/25
D Repeat 8/25
D Repeat 9/25
D Repeat 10/25
D Repeat 11/25
D Repeat 12/25
D Repeat 13/25
D Repeat 14/25
D Repeat 15/25
D Repeat 16/25
D Repeat 17/25
D Repeat 18/25
D Repeat 19/25
D Repeat 20/25
D Repeat 21/25
D Repeat 22/25
D Repeat 23/25
D Repeat 24/25
D Repeat 25/25

Completed Variable Hunting for D

Starting Variable Hunting for M
M Repeat 1/25
M Repeat 2/25
M Repeat 3/25
M Repeat 4/25
M Repeat 5/25
M Repeat 6/25
M Repeat 7/25
M Repeat 8/25
M Repeat 9/25
M Repeat 10/25
M Repeat 11/25
M Repeat 12/25
M Repeat 13/25
M Repeat 14/25
M Repeat 15/25
M Repeat 16/25
M Repeat 17/25
M Repeat 18/25
M Repeat 19/25
M Repeat 20/25
M Repeat 21/25
M Repeat 22/25
M Repeat 23/25
M Repeat 24/25
M Repeat 25/25

Completed Variable Hunting for M

Starting Variable Hunting for DM
DM Repeat 1/25
DM Repeat 2/25
DM Repeat 3/25
DM Repeat 4/25
DM Repeat 5/25
DM Repeat 6/25
DM Repeat 

In [15]:
print("\nTop features in D:")
print(list(cox_results_D["importance"].items())[:10])

print("\nTop features in M:")
print(list(cox_results_M["importance"].items())[:10])

print("\nTop features in DM:")
print(list(cox_results_DM["importance"].items())[:10])


Top features in D:
[('HGB', 0.24556054537923422), ('EAr_D', 0.16089637045404281), ('NYHA class', 0.15639850402376745), ('CO_D', 0.14377932953998726), ('Renal failure', 0.10103398730532649), ('TVr_D', 0.0779722276306294), ('DM', 0.07566269670871707), ('AF', 0.0700312195136274), ('PH', 0.047941213073106594), ('Alcohol', 0.039821776364849876)]

Top features in M:
[('EAr_S', 0.16797569385814942), ('TVr_S', 0.12120800294145631), ('LVEDP_S', 0.07241603181000228), ('C_PA', 0.03351899664358485), ('R_tSA', 0.025343570149917102), ('AVr_S', 0.020213797492741725), ('LVpower_S', 0.016096608858478797), ('PASP_S', 0.014043957997193492), ('k_act_LV', 0.00875558478315838)]

Top features in DM:
[('HGB', 0.2558728974093221), ('NYHA class', 0.16075233942284012), ('CO_D', 0.1420954438686284), ('EAr_S', 0.1034750978825492), ('Renal failure', 0.09904399572960682), ('DM', 0.07380884144533613), ('AF', 0.07096206288744299), ('EAr_D', 0.06767347664349803), ('PH', 0.04695595273955336), ('AVr_S', 0.04431562132412

In [16]:
cox_results_D
cox_results_M
cox_results_DM

{'frequency': {'Age': 25,
  'Alcohol': 25,
  'Drug': 25,
  'HGB': 25,
  'NYHA class': 25,
  'Usage of Diuretic': 25,
  'PH': 25,
  'AF': 25,
  'DM': 25,
  'Renal failure': 25,
  'CO_D': 25,
  'LAVmax_D': 25,
  'TVr_D': 25,
  'EAr_D': 25,
  'C_PA': 25,
  'TVr_S': 25,
  'AVr_S': 25,
  'EAr_S': 25},
 'importance': {'HGB': 0.2558728974093221,
  'NYHA class': 0.16075233942284012,
  'CO_D': 0.1420954438686284,
  'EAr_S': 0.1034750978825492,
  'Renal failure': 0.09904399572960682,
  'DM': 0.07380884144533613,
  'AF': 0.07096206288744299,
  'EAr_D': 0.06767347664349803,
  'PH': 0.04695595273955336,
  'AVr_S': 0.04431562132412517,
  'TVr_D': 0.038772936619327354,
  'Alcohol': 0.03507902345143333,
  'Drug': 0.02750028009556638,
  'TVr_S': 0.020704737391521802,
  'Age': 0.013784119595859907,
  'Usage of Diuretic': 0.013065064659084363,
  'C_PA': 0.009344211657754039,
  'LAVmax_D': 0.009164426303886206}}

In [17]:
result_D = cox_results_D
result_M = cox_results_M
result_DM = cox_results_DM

In [18]:
# ==========================================
# Extract Feature Groups (Cox)
# ==========================================

def extract_feature_groups(result, n_repeats=25):

    freq = result["frequency"]
    importance = result["importance"]

    # Mandatory features: ≥80% frequency
    mandatory = [
        feature for feature, count in freq.items()
        if count >= 0.8 * n_repeats
    ]

    # Valid features: ≥5 repeats
    valid = [
        feature for feature, count in freq.items()
        if count >= 5
    ]

    # Sorted valid features by importance
    sorted_valid = {
        feature: importance[feature]
        for feature in valid
        if feature in importance
    }

    sorted_valid = dict(
        sorted(sorted_valid.items(),
               key=lambda x: x[1],
               reverse=True)
    )

    return mandatory, valid, sorted_valid


mandatory_D, valid_D, sorted_valid_D = extract_feature_groups(result_D)
mandatory_M, valid_M, sorted_valid_M = extract_feature_groups(result_M)
mandatory_DM, valid_DM, sorted_valid_DM = extract_feature_groups(result_DM)


print("\nMandatory D:", len(mandatory_D))
print("Valid D:", len(valid_D))

print("\nMandatory M:", len(mandatory_M))
print("Valid M:", len(valid_M))

print("\nMandatory DM:", len(mandatory_DM))
print("Valid DM:", len(valid_DM))


Mandatory D: 11
Valid D: 11

Mandatory M: 9
Valid M: 9

Mandatory DM: 18
Valid DM: 18


In [19]:
# ==========================================
# Cox Model Training Function
# ==========================================

def train_cox_model(dataset, features, random_state):

    df = dataset[features + ["time", "event"]]

    cph = CoxPHFitter()

    cph.fit(
        df,
        duration_col="time",
        event_col="event"
    )

    risk_scores = cph.predict_partial_hazard(df)

    cindex = concordance_index(
        df["time"],
        -risk_scores,
        df["event"]
    )

    return cindex

In [20]:
# ==========================================
# Cox Forward Feature Selection
# ==========================================

def find_best_features_cox(dataset, sorted_valid_features, mandatory_features,
                           model_name, max_additional=12, repeats=10):

    print(f"\nSelecting best features for {model_name}")

    selected = mandatory_features.copy()
    remaining = list(sorted_valid_features.keys())

    remaining = [f for f in remaining if f not in selected]

    history = []

    # Evaluate mandatory features first
    scores = []

    for rep in range(repeats):

        cindex = train_cox_model(
            dataset,
            selected,
            RANDOM_STATE_BASE + rep
        )

        scores.append(cindex)

    best_score = np.mean(scores)

    history.append((len(selected), best_score, selected.copy()))

    print(f"Initial features: {len(selected)} | C-index: {best_score:.4f}")

    # Forward selection
    for i in range(max_additional):

        best_feature = None
        best_feature_score = best_score

        for feature in remaining:

            test_features = selected + [feature]

            scores = []

            for rep in range(repeats):

                cindex = train_cox_model(
                    dataset,
                    test_features,
                    RANDOM_STATE_BASE + rep
                )

                scores.append(cindex)

            mean_score = np.mean(scores)

            if mean_score > best_feature_score:

                best_feature_score = mean_score
                best_feature = feature

        if best_feature is None:
            break

        selected.append(best_feature)
        remaining.remove(best_feature)
        best_score = best_feature_score

        history.append((len(selected), best_score, selected.copy()))

        print(f"Added: {best_feature} | Total: {len(selected)} | C-index: {best_score:.4f}")

    return selected, history

In [21]:
selected_D, history_D = find_best_features_cox(
    D_imputed, sorted_valid_D, mandatory_D, "D"
)

selected_M, history_M = find_best_features_cox(
    M_imputed, sorted_valid_M, mandatory_M, "M"
)

selected_DM, history_DM = find_best_features_cox(
    DM_imputed, sorted_valid_DM, mandatory_DM, "DM"
)

print("\nFinal selected features:")
print("D:", len(selected_D))
print("M:", len(selected_M))
print("DM:", len(selected_DM))


Selecting best features for D
Initial features: 11 | C-index: 0.7672

Selecting best features for M
Initial features: 9 | C-index: 0.6742

Selecting best features for DM
Initial features: 18 | C-index: 0.7740

Final selected features:
D: 11
M: 9
DM: 18


In [22]:
# ==========================================
# Cox-compatible split (RSF-style)
# ==========================================

def split_dataset_cox(dataset, selected_features, model_name, test_size=0.3):

    print(f"\nSplitting dataset for {model_name}")

    X = dataset[selected_features]

    y = dataset[["time", "event"]]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=42
    )

    print(f"{model_name} Train shape:", X_train.shape)
    print(f"{model_name} Test shape:", X_test.shape)

    return X_train, X_test, y_train, y_test


# Split
X_train_D, X_test_D, y_train_D, y_test_D = split_dataset_cox(D_imputed, selected_D, "D")
X_train_M, X_test_M, y_train_M, y_test_M = split_dataset_cox(M_imputed, selected_M, "M")
X_train_DM, X_test_DM, y_train_DM, y_test_DM = split_dataset_cox(DM_imputed, selected_DM, "DM")


Splitting dataset for D
D Train shape: (240, 11)
D Test shape: (103, 11)

Splitting dataset for M
M Train shape: (240, 9)
M Test shape: (103, 9)

Splitting dataset for DM
DM Train shape: (240, 18)
DM Test shape: (103, 18)


In [23]:
# ==========================================
# CELL: Optimized Hyperparameter tuning with CV (Cox)
# ==========================================

from sklearn.model_selection import KFold, ParameterGrid
from sklearn.preprocessing import StandardScaler

def tune_cox_cv(X_train, y_train, model_name, n_splits=5):

    print(f"\nHyperparameter tuning with CV for {model_name}")

    # Combine X and y
    train_df_full = pd.concat([X_train, y_train], axis=1)

    # Stronger and safer regularization grid
    param_grid = {
        "penalizer": [0.01, 0.1, 0.5, 1.0, 2.0, 5.0],
        "l1_ratio": [0.5, 0.7, 0.9, 1.0]
    }

    kf = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    best_score = -1
    best_params = None

    for params in ParameterGrid(param_grid):

        fold_scores = []

        for fold, (train_idx, val_idx) in enumerate(kf.split(train_df_full)):

            train_df = train_df_full.iloc[train_idx].copy()
            val_df   = train_df_full.iloc[val_idx].copy()

            # ==========================================
            # Scale features (VERY IMPORTANT for Cox)
            # ==========================================
            scaler = StandardScaler()

            feature_cols = X_train.columns

            train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
            val_df[feature_cols]   = scaler.transform(val_df[feature_cols])

            # ==========================================
            # Train Cox model with stability settings
            # ==========================================
            cph = CoxPHFitter(
                penalizer=params["penalizer"],
                l1_ratio=params["l1_ratio"]
            )

            try:

                cph.fit(
                    train_df,
                    duration_col="time",
                    event_col="event",
                    show_progress=False
                )

                risk_scores = cph.predict_partial_hazard(val_df)

                cindex = concordance_index(
                    val_df["time"],
                    -risk_scores,
                    val_df["event"]
                )

                fold_scores.append(cindex)

            except:

                # Skip unstable parameter combinations
                continue

        if len(fold_scores) == 0:
            continue

        mean_score = np.mean(fold_scores)
        std_score  = np.std(fold_scores)

        print(f"{params} → Mean CV C-index: {mean_score:.4f} (±{std_score:.4f})")

        if mean_score > best_score:

            best_score = mean_score
            best_params = params

    print(f"\nBest params for {model_name}:")
    print(best_params)
    print(f"Best CV C-index: {best_score:.4f}")

    return best_params

In [24]:
best_params_D  = tune_cox_cv(X_train_D,  y_train_D,  "D")
best_params_M  = tune_cox_cv(X_train_M,  y_train_M,  "M")
best_params_DM = tune_cox_cv(X_train_DM, y_train_DM, "DM")


Hyperparameter tuning with CV for D
{'l1_ratio': 0.5, 'penalizer': 0.01} → Mean CV C-index: 0.7466 (±0.0744)
{'l1_ratio': 0.5, 'penalizer': 0.1} → Mean CV C-index: 0.7146 (±0.0689)
{'l1_ratio': 0.5, 'penalizer': 0.5} → Mean CV C-index: 0.7549 (±0.0809)
{'l1_ratio': 0.5, 'penalizer': 1.0} → Mean CV C-index: 0.7580 (±0.0788)
{'l1_ratio': 0.5, 'penalizer': 2.0} → Mean CV C-index: 0.7583 (±0.0795)
{'l1_ratio': 0.5, 'penalizer': 5.0} → Mean CV C-index: 0.7583 (±0.0795)
{'l1_ratio': 0.7, 'penalizer': 0.01} → Mean CV C-index: 0.7464 (±0.0755)
{'l1_ratio': 0.7, 'penalizer': 0.1} → Mean CV C-index: 0.6574 (±0.0608)
{'l1_ratio': 0.7, 'penalizer': 0.5} → Mean CV C-index: 0.7577 (±0.0789)
{'l1_ratio': 0.7, 'penalizer': 1.0} → Mean CV C-index: 0.7583 (±0.0787)
{'l1_ratio': 0.7, 'penalizer': 2.0} → Mean CV C-index: 0.7583 (±0.0795)
{'l1_ratio': 0.7, 'penalizer': 5.0} → Mean CV C-index: 0.7583 (±0.0795)
{'l1_ratio': 0.9, 'penalizer': 0.01} → Mean CV C-index: 0.7451 (±0.0738)
{'l1_ratio': 0.9, 'penal

In [26]:
# ==========================================
# Install scikit-survival
# ==========================================

!pip install scikit-survival --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 15.2 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 120.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 17.7 MB/s eta 0:00:00


In [27]:
# ==========================================
# CELL: Final Cox training and testing (Optimized)
# ==========================================

from sklearn.preprocessing import StandardScaler
from sksurv.metrics import cumulative_dynamic_auc, integrated_brier_score
from sksurv.util import Surv

def train_and_evaluate_test_cox(X_train, y_train,
                                X_test, y_test,
                                best_params, model_name):

    print(f"\nFinal Cox training and testing for {model_name}")

    # ==========================================
    # Scale features (CRITICAL)
    # ==========================================
    scaler = StandardScaler()

    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )

    X_test_scaled = pd.DataFrame(
        scaler.transform(X_test),
        columns=X_test.columns,
        index=X_test.index
    )

    # Combine scaled X and survival data
    train_df = pd.concat([X_train_scaled, y_train], axis=1)
    test_df  = pd.concat([X_test_scaled, y_test], axis=1)

    # ==========================================
    # Train Cox model
    # ==========================================
    cph = CoxPHFitter(
        penalizer=best_params["penalizer"],
        l1_ratio=best_params["l1_ratio"]
    )

    cph.fit(
        train_df,
        duration_col="time",
        event_col="event"
    )

    # ==========================================
    # Risk scores
    # ==========================================
    risk_scores = cph.predict_partial_hazard(test_df).values.flatten()

    # ==========================================
    # C-index
    # ==========================================
    cindex = concordance_index(
        test_df["time"],
        -risk_scores,
        test_df["event"]
    )

    # ==========================================
    # Convert to sksurv format
    # ==========================================
    y_train_sksurv = Surv.from_dataframe("event", "time", train_df)
    y_test_sksurv  = Surv.from_dataframe("event", "time", test_df)

    # ==========================================
    # Time grid
    # ==========================================
    max_time = min(train_df["time"].max(), test_df["time"].max())

    valid_times = np.linspace(
        test_df["time"].min() + 1e-6,
        max_time * 0.99,
        20
    )

    # ==========================================
    # iAUC
    # ==========================================
    auc_times, auc_scores = cumulative_dynamic_auc(
        y_train_sksurv,
        y_test_sksurv,
        risk_scores,
        valid_times
    )

    iauc = np.mean(auc_scores)

    # ==========================================
    # Brier score
    # ==========================================
    surv_funcs = cph.predict_survival_function(test_df, times=valid_times)

    surv_matrix = surv_funcs.T.values

    brier = integrated_brier_score(
        y_train_sksurv,
        y_test_sksurv,
        surv_matrix,
        valid_times
    )

    # ==========================================
    # Print results
    # ==========================================
    print(f"\n{model_name} TEST Results:")
    print(f"C-index: {cindex:.4f}")
    print(f"iAUC:    {iauc:.4f}")
    print(f"Brier:   {brier:.4f}")

    return cph, cindex, iauc, brier, scaler

In [28]:
model_D, cindex_D, iauc_D, brier_D, scaler_D = train_and_evaluate_test_cox(
    X_train_D, y_train_D,
    X_test_D, y_test_D,
    best_params_D,
    "D"
)

model_M, cindex_M, iauc_M, brier_M, scaler_M = train_and_evaluate_test_cox(
    X_train_M, y_train_M,
    X_test_M, y_test_M,
    best_params_M,
    "M"
)

model_DM, cindex_DM, iauc_DM, brier_DM, scaler_DM = train_and_evaluate_test_cox(
    X_train_DM, y_train_DM,
    X_test_DM, y_test_DM,
    best_params_DM,
    "DM"
)


Final Cox training and testing for D

D TEST Results:
C-index: 0.7462
iAUC:    0.7505
Brier:   0.1845

Final Cox training and testing for M

M TEST Results:
C-index: 0.6492
iAUC:    0.6899
Brier:   0.1845

Final Cox training and testing for DM

DM TEST Results:
C-index: 0.7709
iAUC:    0.7855
Brier:   0.1845


In [30]:
test_data = pd.read_excel("/kaggle/input/datasets/sweta2000/umhs-dataset/Testing.xlsx")

# Apply SAME preprocessing
test_data["Race"] = np.where(test_data["Race"] == "Caucasian", 1, 0)
test_data["Smoking"] = np.where(test_data["Smoking"] == "Current", 1, 0)
test_data["Alcohol"] = np.where(test_data["Alcohol"] == "Yes", 1, 0)
test_data["Drug"] = np.where(test_data["Drug"] == "Yes", 1, 0)

test_data["time"] = test_data["Duration"]
test_data["event"] = test_data["EndEvent"]
a
test_data.drop(columns=["Duration", "EndEvent"], inplace=True)

In [31]:
# ==========================================
# CELL: Final Cox training and evaluation on EXTERNAL dataset
# ==========================================

def train_and_evaluate_external_cox(train_dataset, test_dataset,
                                    selected_features, best_params,
                                    model_name):

    print(f"\nFINAL Cox Training and External Evaluation: {model_name}")

    # ==========================================
    # Select features
    # ==========================================
    X_train = train_dataset[selected_features]
    y_train = train_dataset[["time", "event"]]

    X_test = test_dataset[selected_features]
    y_test = test_dataset[["time", "event"]]

    # ==========================================
    # Imputation
    # ==========================================
    imputer = KNNImputer(n_neighbors=5)

    X_train = pd.DataFrame(
        imputer.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )

    X_test = pd.DataFrame(
        imputer.transform(X_test),
        columns=X_test.columns,
        index=X_test.index
    )

    # ==========================================
    # Scaling (important for Cox)
    # ==========================================
    scaler = StandardScaler()

    X_train = pd.DataFrame(
        scaler.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )

    X_test = pd.DataFrame(
        scaler.transform(X_test),
        columns=X_test.columns,
        index=X_test.index
    )

    train_df = pd.concat([X_train, y_train], axis=1)
    test_df  = pd.concat([X_test, y_test], axis=1)

    # ==========================================
    # Train Cox model
    # ==========================================
    cph = CoxPHFitter(
        penalizer=best_params["penalizer"],
        l1_ratio=best_params["l1_ratio"]
    )

    cph.fit(train_df, duration_col="time", event_col="event")

    # ==========================================
    # Risk scores
    # ==========================================
    risk_scores = cph.predict_partial_hazard(test_df).values.flatten()

    # ==========================================
    # C-index
    # ==========================================
    cindex = concordance_index(
        test_df["time"],
        -risk_scores,
        test_df["event"]
    )

    # ==========================================
    # Convert to sksurv format
    # ==========================================
    y_train_sksurv = Surv.from_dataframe("event", "time", train_df)
    y_test_sksurv  = Surv.from_dataframe("event", "time", test_df)

    # ==========================================
    # Time grid
    # ==========================================
    max_time = min(train_df["time"].max(), test_df["time"].max())

    valid_times = np.linspace(
        test_df["time"].min() + 1e-6,
        max_time * 0.99,
        20
    )

    # ==========================================
    # iAUC
    # ==========================================
    auc_times, auc_scores = cumulative_dynamic_auc(
        y_train_sksurv,
        y_test_sksurv,
        risk_scores,
        valid_times
    )

    iauc = np.mean(auc_scores)

    # ==========================================
    # Brier score
    # ==========================================
    surv_funcs = cph.predict_survival_function(test_df, times=valid_times)

    surv_matrix = surv_funcs.T.values

    brier = integrated_brier_score(
        y_train_sksurv,
        y_test_sksurv,
        surv_matrix,
        valid_times
    )

    # ==========================================
    # Print results (same format as RSF)
    # ==========================================
    print(f"\n{model_name} External Results:")
    print(f"C-index: {cindex:.4f}")
    print(f"iAUC:    {iauc:.4f}")
    print(f"Brier:   {brier:.4f}")

    return cph, scaler, cindex, iauc, brier

In [32]:
model_D, scaler_D, cindex_D, iauc_D, brier_D = train_and_evaluate_external_cox(
    data, test_data,
    selected_D,
    best_params_D,
    "D"
)

model_M, scaler_M, cindex_M, iauc_M, brier_M = train_and_evaluate_external_cox(
    data, test_data,
    selected_M,
    best_params_M,
    "M"
)

model_DM, scaler_DM, cindex_DM, iauc_DM, brier_DM = train_and_evaluate_external_cox(
    data, test_data,
    selected_DM,
    best_params_DM,
    "DM"
)


FINAL Cox Training and External Evaluation: D

D External Results:
C-index: 0.6516
iAUC:    0.6863
Brier:   0.2276

FINAL Cox Training and External Evaluation: M

M External Results:
C-index: 0.6645
iAUC:    0.6551
Brier:   0.2276

FINAL Cox Training and External Evaluation: DM

DM External Results:
C-index: 0.6629
iAUC:    0.6847
Brier:   0.2276
